### UROP Research Log — July 5, 2026

### Objective
Reconstruct BXM index value at 15:45 ET for every trading day in 2022,
and compare against PBP ETF 15:45 ET mid quote to compute synchronized TE.

### Why
Stage 1 (Standard Close) gave 4.80% annualized TE — inflated due to:
1. Low-quality Nasdaq PBP data
2. Timestamp mismatch: PBP at 4:00 PM ET vs BXM at ~4:15 PM ET

First attempt at Stage 2 (PBP at 15:45, BXM at daily close) gave 4.92% —
even worse, because the timing gap actually increased from 15 min to 30 min.

True fix requires synchronizing BOTH sides to 15:45 ET:
- PBP 15:45: mid quote from Cboe Equity EOD Summary ✅
- BXM 15:45: reconstruct from Cboe SPX options tick data

### What is BXM at 15:45 ET?

On any given day, the BXM portfolio consists of:
1. Long SPX position
2. Short 1 ATM call option (sold on the last roll date)

BXM portfolio value at 15:45 ET:
    BXM_1545 = SPX_1545 - Call_option_mid_1545

Where:
- SPX_1545 = underlying_bid from Cboe tick data at 15:45 ET
- Call_option_mid_1545 = (best_bid + best_ask) / 2 of the
  current ATM option at 15:45 ET

Daily BXM return (non-roll day):
    R_BXM_1545 = log(BXM_1545_t / BXM_1545_{t-1})

On roll dates (3rd Friday):
- Settle expiring option at SOQ (Special Opening Quotation)
- Sell new ATM option at 15:45 mid quote
- Start tracking new option

### Data Sources
- PBP 15:45 mid quote: Cboe Equity EOD Summary
  ../data/raw/Equity EOD Summary/UnderlyingEOD_YYYY-MM.csv
- SPX 15:45 level + option mid quote: Cboe SPX options tick data
  ../data/raw/cboe_spx_2022/UnderlyingOptionsTradesCalcs_YYYY-MM-DD.csv
- SOQ values: cboe.com/index_settlement_values/ (12 values, 2022 roll dates)
- BXM daily close: ../data/raw/BXM_History.csv (for reference/validation)

### Revised Approach (after initial attempt)

Initial approach: reconstruct BXM_1545 = SPX_1545 - Call_mid_1545 from tick data
Result: TE jumped to 15.29%, R²=0.55 — much worse than Stage 1 (4.80%, R²=0.92)
Root cause: SPX option NBBO (best_bid/best_ask) is highly volatile intraday,
causing option_mid to swing dramatically and making BXM_1545 returns too noisy.

Revised approach:
- PBP side: Cboe Equity EOD best_bid_1545/best_ask_1545 → clean official data ✅
- BXM side: Official BXM_History.csv daily close → Cboe-computed, authoritative ✅

Key insight: the primary improvement in Stage 2 is PBP data quality
(Nasdaq → Cboe Equity EOD), not BXM timestamp reconstruction.

Remaining limitation: PBP at 15:45 vs BXM at ~16:15 (30 min gap remains).
This will be documented as an open issue pending future data acquisition.

### Known Limitations
- SPX_1545 is approximated via underlying_bid (options tick data),
  not the actual SPX index feed — may introduce small bias
- Option mid quote at 15:45 may be stale if no trades near that time
- Roll date timing assumption: BXM sells new option at 15:45 on roll date

### Notes
- Professor Dodson (Zoom, June 26): "you can get the mark-to-market
  on the options in your pretend portfolio at 3:45 for every day"
- Mid quote = (best_bid + best_ask) / 2 is the correct MTM convention
- VWAP is PBP's cost of doing business, not our BXM construction price

Note : 2022-11-25 was Black Friday. Market closed early at 1:00PM ET.

In [18]:
# -- Load Data --
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import glob

# PBP Cboe EOD
eod_path = "../data/raw/Equity EOD Summary/"
files = sorted(glob.glob(eod_path + "UnderlyingEOD_*.csv"))
pbp_eod = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
pbp_eod['quote_date'] = pd.to_datetime(pbp_eod['quote_date'])
pbp_2022 = pbp_eod[pbp_eod['quote_date'].dt.year == 2022].copy()
pbp_2022 = pbp_2022.sort_values('quote_date').reset_index(drop=True)
pbp_2022['mid_1545'] = (pbp_2022['best_bid_1545'] + pbp_2022['best_ask_1545']) / 2

# BXM
bxm = pd.read_csv("../data/raw/BXM_History.csv")
bxm['DATE'] = pd.to_datetime(bxm['DATE'])
bxm_2022 = bxm[bxm['DATE'].dt.year == 2022].copy()
bxm_2022 = bxm_2022.rename(columns={'DATE': 'date', 'BXM': 'bxm_close'})

# Merge
df = pd.merge(
    pbp_2022[['quote_date', 'mid_1545']].rename(columns={'quote_date': 'date'}),
    bxm_2022[['date', 'bxm_close']],
    on='date', how='inner'
).sort_values('date').reset_index(drop=True)

print(f"Merged rows: {len(df)}")
print(df.head(3))

Merged rows: 251
        date  mid_1545  bxm_close
0 2022-01-03     23.16    1778.59
1 2022-01-04     23.16    1778.63
2 2022-01-05     23.08    1767.15
